# Week 2: Text Cleaning and Normalization

This notebook analyzes the Week 1 MLS listing sample and evaluates the text cleaning pipeline.

The goal is to identify common formatting issues in listing remarks, apply normalization methods, and compare the text before and after cleaning.

In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/listing_sample.csv")

print("Shape:", df.shape)
df.head()

Shape: (1000, 7)


,L_ListingID,L_Address,L_City,beds,baths,price,remarks
0,1189451542,36612 Desert Tortoise Road,Murrieta,3.0,3.0,635000,"Welcome to 36612 Desert Tortoise Road, a beaut..."
1,1159634800,904 Amistad Court C,El Cajon,2.0,2.0,409000,PRICE REDUCED! on this IMMACULATE turnkey cond...
2,1144651476,1255 Delresto Drive,Beverly Hills,4.0,4.0,14495000,"Majestic Gated Italian Villa, An Architectural..."
3,1179077960,16855 Redwood Lodge Road,Los Gatos,4.0,3.0,1195000,Tucked into the trees of the Santa Cruz Mounta...
4,1190215696,5252 Agnes 2,Valley Village,2.0,2.0,599000,A magnificently remodeled corner unis with 3 (...


## Data Profiling

Before cleaning the listing remarks, the dataset is profiled to identify common text issues such as missing values, HTML tags, price mentions, and unusual spacing.

In [2]:
import re

remarks = df["remarks"]

print("Number of listings:", len(remarks))
print("Null remarks:", remarks.isnull().sum())
print("Average remark length:", remarks.str.len().mean())

html_count = remarks.fillna("").str.contains(r"<[^>]+>", regex=True).sum()
price_count = remarks.fillna("").str.contains(r"\$\s*\d", regex=True).sum()
extra_space_count = remarks.fillna("").str.contains(r"\s{2,}", regex=True).sum()

print("Listings containing HTML:", html_count)
print("Listings containing price mentions:", price_count)
print("Listings containing repeated spaces:", extra_space_count)

Number of listings: 1000
Null remarks: 0
Average remark length: 1326.651
Listings containing HTML: 0
Listings containing price mentions: 68
Listings containing repeated spaces: 500


## Abbreviation Analysis

Common real estate abbreviations are identified in the listing remarks to determine which terms should be expanded during text cleaning.

This helps ensure that the abbreviation dictionary reflects patterns that actually appear in the dataset.

In [3]:
abbreviations = [
    "br", "bd", "bdr", "bdrm",
    "ba", "bth",
    "sqft", "sq ft", "sf",
    "w/", "w/o",
    "mbr",
    "fp",
    "gar",
    "ac", "a/c",
    "hoa",
    "apt",
    "condo",
    "yr", "yrs",
    "mo", "mos",
    "pkg", "prkg",
    "rm",
    "lvl",
    "flr",
    "kit",
    "lr",
    "dr",
    "fam rm",
    "laund"
]

abbreviation_counts = {}

for abbrev in abbreviations:
    pattern = rf"(?<![A-Za-z]){re.escape(abbrev)}(?![A-Za-z])"

    count = (
        remarks.fillna("")
        .str.contains(pattern, case=False, regex=True)
        .sum()
    )

    abbreviation_counts[abbrev] = count

abbreviation_counts = {
    abbreviation: count
    for abbreviation, count in abbreviation_counts.items()
    if count > 0
}

sorted(
    abbreviation_counts.items(),
    key=lambda x: x[1],
    reverse=True
)

[('hoa', 109),
 ('sq ft', 81),
 ('condo', 80),
 ('ac', 33),
 ('a/c', 30),
 ('sqft', 25),
 ('ba', 18),
 ('sf', 18),
 ('w/', 16),
 ('br', 14),
 ('bd', 9),
 ('dr', 5),
 ('bdrm', 1),
 ('fp', 1),
 ('mo', 1)]

## Apply the Text Cleaning Pipeline

The `TextCleaner` is applied to all listing remarks to standardize formatting, expand abbreviations, normalize prices and measurements, and remove unnecessary spacing.

The cleaned text is stored in a new column so the original and cleaned versions can be compared.

In [10]:
import importlib
import scripts.text_cleaning

importlib.reload(scripts.text_cleaning)

from scripts.text_cleaning import TextCleaner

cleaner = TextCleaner()

df["cleaned_remarks"] = df["remarks"].apply(cleaner.clean_text)

## Before and After Examples

The examples below show how the cleaning pipeline transforms raw MLS remarks into a more standardized format.

In [11]:
for i in range(5):
    print("BEFORE:")
    print(df.loc[i, "remarks"])

    print("\nAFTER:")
    print(df.loc[i, "cleaned_remarks"])

    print("\n" + "-" * 80 + "\n")

BEFORE:
Welcome to 36612 Desert Tortoise Road, a beautifully designed Lennar home built in 2024 and tucked inside the gated Willow Springs community. Offering approximately 2,025 square feet of thoughtfully designed living space, this 3-bedroom, 2.5-bath home combines the feel of newer construction with a floor plan created for the way people live today. Step inside to a bright, open-concept main level where the Great Room, dining area and kitchen flow seamlessly together—creating an inviting space for everyday living, entertaining and gathering. The kitchen is highlighted by quartz countertops, a generous island/breakfast bar and abundant workspace, while sliding doors extend the living area outdoors. Upstairs, the versatile loft creates valuable additional space for a home office, media room, playroom, fitness area or second family room. All three bedrooms are conveniently located upstairs, including a spacious primary suite with a private bath and walk-in closet. An upstairs laundry

## Save Cleaned Dataset

The cleaned listing remarks are saved as a new processed dataset so they can be reused in later NLP tasks such as entity extraction and query parsing.

In [12]:
df.to_csv(
    "../data/processed/listing_sample_cleaned.csv",
    index=False
)

print("Saved cleaned dataset:")
print("../data/processed/listing_sample_cleaned.csv")
print("Shape:", df.shape)

Saved cleaned dataset:
../data/processed/listing_sample_cleaned.csv
Shape: (1000, 8)
